In [ ]:
import sys
print(sys.executable)

!{sys.executable} -m pip install ultralytics
!{sys.executable} -m pip install sahi

## Imports

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.slicing import slice_image
from scipy.stats import variation

if "notebooks" in os.getcwd():
    os.chdir("../../../")

## Helper Functions

### Calculation of IoU

In [ ]:
def get_iou(box1, box2):
    """Calculates Intersection over Union (IoU) between two boxes [x1, y1, x2, y2]"""
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return intersection_area / float(area1 + area2 - intersection_area)

### Calculation of True Pos, False Pos, False Neg

In [ ]:
def calculate_tp_fp_fn(preds, gts, iou_threshold=0.5):
    """Matches predictions to ground truth to find TP, FP, and FN"""
    tp = 0
    fp = 0
    matched_gt_indices = set()

    for p_box in preds:
        best_iou = 0
        best_gt_idx = -1

        for i, g_box in enumerate(gts):
            if i in matched_gt_indices:
                continue
            iou = get_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i

        if best_iou >= iou_threshold:
            tp += 1
            matched_gt_indices.add(best_gt_idx)
        else:
            fp += 1

    fn = len(gts) - len(matched_gt_indices)
    return tp, fp, fn

### Extracting Seed Dimensions

In [ ]:
def extract_morphology(sahi_result, ppm=1.0):
    """
    Extracts dimensions from SAHI OBB predictions.
    """

    seeds_data = []

    for pred in sahi_result.object_prediction_list:
        b = pred.bbox

        # Calculate width and height using coordinate subtraction
        w_px = b.maxx - b.minx
        h_px = b.maxy - b.miny

        # Avoid division by zero
        if w_px <= 0 or h_px <= 0:
            continue

        # 'Length' is always the larger dimension and 'width' the shorter
        length_px = max(w_px, h_px)
        width_px = min(w_px, h_px)

        area_px2 = length_px * width_px
        aspect_ratio = length_px / width_px

        seeds_data.append({
            'length_px': length_px,
            'width_px': width_px,
            'area_px2': area_px2,
            'aspect_ratio': aspect_ratio
        })

    return pd.DataFrame(seeds_data)

### Seed Variability Calculation

In [ ]:
def calculate_seed_viability(df):
    """
    Categorizes seeds based on the 30% size rule.
    """
    # Use Median to establish a more stable 'typical' seed size
    baseline_area = df['area_mm2'].median()
    threshold = baseline_area * 0.30

    # Categorize
    df['status'] = np.where(df['area_mm2'] <= threshold, 'Aborted', 'Active')

    # Summary Stats
    counts = df['status'].value_counts().to_dict()
    active_count = counts.get('Active', 0)
    aborted_count = counts.get('Aborted', 0)

    return active_count, aborted_count, threshold

## Model Training & Image Slicing

In [ ]:
# Create a new YOLO26n-OBB model from scratch
# model = YOLO("yolo26n-obb.yaml")
model = YOLO("yolo26n-obb.pt") # Using a pre-trained model

# Train the model on the dataset
train_results = model.train(
    data="data/seed/data.yaml",
    epochs=100,
    imgsz=768,       # The size of the "window" the CPU looks at
    batch=2,         # Low batch for CPU stability
    crop_fraction=0.2,  # FOCUS: This tells YOLO to crop a small area
                        # instead of resizing the whole 5472px image.
                        # 0.2 means it takes a 20% window of the original size.
    mosaic=0.0,         # Turn off mosaic for very small objects (seeds)
    close_mosaic=0,     # Keeps the detail sharp
    patience=50,
    plots=True
)

## Load model to SAHI

In [ ]:
# Load the model into SAHI's wrapper
# We point to the 'best.pt' created by the training above
best_model_path = os.path.join(train_results.save_dir, 'weights/best.pt')

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=best_model_path,
    confidence_threshold=0.5,
    device="cpu", # Change to "cuda:0" if GPU
)

# Get the list of images in your val folder
val_img_dir = "data/seed/images/val/"
val_images = [f for f in os.listdir(val_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

if not val_images:
    raise FileNotFoundError("No images found in the validation folder!")

# Define the missing directory variable
export_base_dir = "sahi_results/"

# Make sure the folder actually exists so Python doesn't complain
os.makedirs(export_base_dir, exist_ok=True)


## SAHI Inference 

In [ ]:
# Variables for the calculations of results.
total_error = 0
total_gt = 0
total_tp = 0
total_fp = 0
total_fn = 0

print(f"Starting batch processing on {len(val_images)} images...")

# Loop through every image
for img_name in val_images:
    target_image_path = os.path.join(val_img_dir, img_name)

    # --- Load ground truth labels ---
    label_path = target_image_path.replace("images", "labels") \
                                  .replace(".jpg", ".txt") \
                                  .replace(".png", ".txt") \
                                  .replace(".jpeg", ".txt")

    # load ground truth
    gt_boxes = []
    if os.path.exists(label_path):
        # Image dimensions to denormalize the coordinates
        img_cv = cv2.imread(target_image_path)
        img_h, img_w = img_cv.shape[:2]
        with open(label_path, "r") as f:
            for line in f.readlines():
                values = list(map(float, line.strip().split()))
                # We are working with OBB labels that have 1 class + 8 coordinates
                if len(values) >= 9:
                    coords = values[1:]
                    xs = [c * img_w for c in coords[0::2]]
                    ys = [c * img_h for c in coords[1::2]]
                    gt_boxes.append([min(xs), min(ys), max(xs), max(ys)])

    # Run Sliced Prediction
    result = get_sliced_prediction(
        target_image_path,
        detection_model,
        slice_height=768,
        slice_width=768,
        overlap_height_ratio=0.4, # Overlap between SAHI slices, height-wise
        overlap_width_ratio=0.4,  # Same as above but width-wise
        postprocess_type="NMS",
        postprocess_match_metric="IOU",
        postprocess_match_threshold=0.25
    )

    # Convert SAHI results to coordinate list for matching gt_boxes list
    preds = []
    for pred in result.object_prediction_list:
        b = pred.bbox
        preds.append([b.minx, b.miny, b.maxx, b.maxy])

    # Calculate True Pos (TP), False Pos (FP), False Neg (FN)
    tp, fp, fn = calculate_tp_fp_fn(preds, gt_boxes, iou_threshold=0.4) # Using 0.4 threshold for small seeds

    total_tp += tp
    total_fp += fp
    total_fn += fn

    expected_count = len(gt_boxes)
    detected_count = len(preds)

    diff = detected_count - expected_count
    total_error += abs(diff)
    total_gt += expected_count

    # Difference percentage. Avoid division by zero if an image has no labels
    pct_diff = (diff / expected_count * 100) if expected_count > 0 else 0
    print(f"{img_name}: TP={tp}, FP={fp}, FN={fn}, Diff={diff} ({pct_diff:.1f}%)")

    # Save visuals
    output_subdir = os.path.join(export_base_dir, os.path.splitext(img_name)[0])
    result.export_visuals(
        export_dir=output_subdir,
        hide_labels=True,         # Hide label class on image
        hide_conf=False,          # Hide confidence label on image
        rect_th=2                 # Bounding box thickness
    )


    # Check if difference is within the acceptable range (+- 10% of expected seed count)
    expected_range = 10 # 10% difference based on discussion with Maria
    range_acceptance = "WITHIN ACCEPTABLE RANGE" if abs(diff) <= expected_range else "OUTSIDE OF ACCEPTABLE RANGE"
    print(range_acceptance)

    # Calculate seed morphology to count aborted vs active seeds
    df_seeds = extract_morphology(result)

    if not df_seeds.empty:
        active, aborted, thresh = calculate_seed_viability(df_seeds)

        print(f"Active seeds: {active}")
        print(f"Aborted seeds: {aborted}")
    else:
        print("No seeds detected in this image.")

    print("-" * 30)

# Final Summary Calculation
mae = total_error / len(val_images) if val_images else 0

print("\n==== FINAL RESULTS ====")
pct_total_error = (total_error / total_gt * 100) if expected_count > 0 else 0
print(f"Total difference: {total_error} ({pct_total_error:.1f}%)")
print(f"Mean Absolute Error (MAE): {mae:.2f} seeds per image")

## Detection Metrix

In [ ]:
# Calculate Precision, Recall, F1-Score
precision = total_tp / (total_tp + total_fp + 1e-6)
recall = total_tp / (total_tp + total_fn + 1e-6)
f1 = 2 * precision * recall / (precision + recall + 1e-6)

print("\n==== DETECTION METRICS ====")
print(f"Total TP: {total_tp}")
print(f"Total FP: {total_fp}")
print(f"Total FN: {total_fn}")
print(f"Precision: {precision:.3f}") # Goal: 0.90-0.95
print(f"Recall:    {recall:.3f}")    # Goal: 0.80-0.90
print(f"F1-score:  {f1:.3f}")        # Goal: 0.85+

## Uncertainty Heatmaps

In [ ]:
results = model("data/seed/images/val/", save=True)

# color code by confidence
def get_color(score):
    if score > 0.7:
        return (0, 255, 0) # high confidence: green
    elif score > 0.4:
        return (0, 255, 255) # medium confidence: yellow
    else:
        return (0, 0, 255) # low confidence: red

for r in results:

    if r.obb is None:
        continue

    img = r.orig_img.copy()
    obb = r.obb

    for box, score, cls in zip(obb.xyxyxyxy, obb.conf, obb.cls):
        pts = np.array(box, dtype=np.int32)

        color = get_color(float(score))
        cv2.polylines(img, [pts], True, color, 2)

    # convert BGR to RGB for matplotlib
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
# 1. Create a blank accumulation mask
heatmap_mask = np.zeros((r.orig_img.shape[0], r.orig_img.shape[1]), dtype=np.float32)

for box, score in zip(r.obb.xyxyxyxy, r.obb.conf):
    # Get the center of the OBB
    center_x = int(box[:, 0].mean())
    center_y = int(box[:, 1].mean())

    # Add intensity at the center point (scaled by confidence)
    heatmap_mask[center_y, center_x] += float(score)

# 2. Apply Gaussian Blur to spread the "heat"
heatmap_mask = cv2.GaussianBlur(heatmap_mask, (51, 51), 0)

# 3. Normalize and Apply Colormap
heatmap_mask = cv2.normalize(heatmap_mask, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

# Alternative: highlight low-confidence detections instead of high-confidence ones
# heatmap_mask[center_y, center_x] += (1 - float(score))

heatmap_color = cv2.applyColorMap(heatmap_mask, cv2.COLORMAP_JET)

# 4. Overlay on original image
alpha = 0.5
overlay = cv2.addWeighted(r.orig_img, 1 - alpha, heatmap_color, alpha, 0)

## Intelligent Debris Filtering